# 개별종목 조합A — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합A 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합A의 피처 값만 지정합니다.
import json

COMBINATION = 'A'
FEATURE_COLUMNS = (
    'sma_gap_5_20',
    'sma_gap_20_60',
    'rsi_14',
    'macd_hist_ratio',
    'bb_bandwidth',
    'bb_position',
    'atr_ratio',
    'hv_20',
    'vol_ratio_20',
    'obv_slope_20',
    'daily_return',
    'five_day_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합A 피처: ('sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'macd_hist_ratio', 'bb_bandwidth', 'bb_position', 'atr_ratio', 'hv_20', 'vol_ratio_20', 'obv_slope_20', 'daily_return', 'five_day_return')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,NaN,750,20140217,20140514,0.4466,0.5012,-0.0546,0.3428,0.2255,0.3128
1,2,balanced,980,20150123,20150421,0.3659,0.3978,-0.0320,0.3497,0.3109,0.3405
2,3,balanced,1210,20151228,20160328,0.3587,0.3762,-0.0175,0.3581,0.3885,0.3679
3,4,balanced,1439,20161202,20170228,0.4270,0.4617,-0.0348,0.3713,0.2702,0.3434
4,5,balanced,1669,20171113,20180207,0.3699,0.3901,-0.0202,0.3578,0.3383,0.3548
5,6,balanced,1899,20181024,20190118,0.3872,0.3725,0.0148,0.3873,0.4148,0.3960
6,7,balanced,2129,20190930,20191224,0.4259,0.4781,-0.0523,0.3504,0.2572,0.3301
7,8,balanced,2359,20200902,20201130,0.3680,0.3476,0.0204,0.3680,0.4356,0.3881
8,9,balanced,2589,20210806,20211105,0.3697,0.3916,-0.0219,0.3592,0.2630,0.3229
9,10,balanced,2818,20220714,20221012,0.3499,0.3454,0.0045,0.3484,0.3144,0.3368


,OOS 폴드 평균
accuracy,0.3853
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0115
macro_f1,0.3618
down_recall,0.3335
core_harmonic_mean,0.3546


재실행 명령: python scripts/run_stock_model_experiment.py
